# 02 — Transformações de DataFrames e Datas

Funções: `trata_data_flexivel`, `normaliza_linha_percentual`, `converte_colunas_data_numericas`, `colunas_por_delimitadores`, `reduz_categorias`, `Conta_dias_uteis`, `contagem_ativos_por_dia`

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')


## 1. `trata_data_flexivel`

Converte um único valor para `pd.Timestamp` aceitando: `DD/MM/YYYY`, `DD/MM/YY`, `YYYY-MM-DD`, data com hora, e serial numérico do Excel (dias desde 1899-12-30).

In [ ]:
from funcoes_data_frames.transformacoes import trata_data_flexivel

exemplos = pd.Series([
    '15/06/2026',        # DD/MM/YYYY
    '15/06/26',          # DD/MM/YY
    '2026-06-15',        # YYYY-MM-DD
    '2026-06-15 14:30',  # com hora -> mantem so a data
    45823,               # serial Excel
    None,                # nulo -> NaT
])

resultado = exemplos.apply(trata_data_flexivel)
pd.DataFrame({'original': exemplos, 'convertido': resultado})


## 2. `normaliza_linha_percentual`

Normaliza cada linha de um DataFrame numérico como % da soma da linha. Ideal para transformar contagens absolutas em representatividade diária.

In [ ]:
from funcoes_data_frames.transformacoes import normaliza_linha_percentual

df_lig = pd.DataFrame({
    'EU_USO':     [150, 200, 130],
    'EU_PAGO':    [ 80,  90,  70],
    'EU_CANCELO': [ 20,  15,  25],
}, index=['2026-06-01', '2026-06-02', '2026-06-03'])

print("Contagens absolutas:")
print(df_lig)

df_pct = normaliza_linha_percentual(df_lig)
print("\nRepresentatividade por dia (%):")
print(df_pct.round(1))
print(f"\nSoma de cada linha: {df_pct.sum(axis=1).values}")  # deve ser [100, 100, 100]


## 3. `converte_colunas_data_numericas`

Detecta colunas cujo nome contenha um padrão (default `'data'`) e converte para datetime. Colunas numéricas são tratadas como serial do Excel; as demais via `pd.to_datetime`.

In [ ]:
from funcoes_data_frames.preparacao_dados import converte_colunas_data_numericas

df_excel = pd.DataFrame({
    'data_pedido':  [45823, 45824, 45825],   # seriais Excel
    'data_entrega': [45830, 45831, 45832],   # seriais Excel
    'valor':        [100.0, 200.0, 150.0],   # numerica normal: nao toca
    'status':       ['OK', 'OK', 'Pendente'],
})

print("Tipos antes:")
print(df_excel.dtypes)

df_conv = converte_colunas_data_numericas(df_excel, padrao='data')

print("\nDepois:")
print(df_conv)
print(df_conv.dtypes)


## 4. `colunas_por_delimitadores`

Explode uma coluna com valores concatenados por delimitador, gerando uma linha por valor (útil para colunas de tags ou serviços múltiplos).

In [ ]:
from funcoes_data_frames.transformacoes import colunas_por_delimitadores

df_tags = pd.DataFrame({
    'cliente_id': [1, 2, 3],
    'servicos':   ['internet;fixo;tv', 'movel', 'internet;movel'],
    'valor_mes':  [150.0, 80.0, 120.0],
})

print("Antes (1 linha por cliente):")
print(df_tags)

df_exp = colunas_por_delimitadores(df_tags, coluna='servicos', delimitador=';')
print("\nDepois (1 linha por servico):")
print(df_exp[['cliente_id', 'value', 'valor_mes']].rename(columns={'value': 'servico'}))


## 5. `reduz_categorias`

Recebe uma Series de contagens e retorna um dicionário mapeando categorias com frequência abaixo do `cutoff` para `'Other'`.

In [ ]:
from funcoes_data_frames.transformacoes import reduz_categorias

freq = pd.Series({
    'Loja':       450,
    'Online':     320,
    'Televendas': 180,
    'Parceiro':    45,
    'WhatsApp':    22,
    'Email':        8,
})

mapeamento = reduz_categorias(freq, cutoff=50)
print("Mapeamento (< 50 ocorrencias -> 'Other'):")
print(mapeamento)

df_canais = pd.DataFrame({'canal': freq.index.repeat(freq.values)})
df_canais['canal_reduzido'] = df_canais['canal'].map(mapeamento)
print(f"\nDistribuicao final:\n{df_canais['canal_reduzido'].value_counts()}")


## 6. `Conta_dias_uteis`

Calcula dias úteis entre duas séries de datas (exclui sábados, domingos e feriados opcionais). Retorna `NaN` para datas inválidas ou `fim < inicio`.

In [ ]:
from ML.supervisionado.series_temporais.transformacoes_temporais import Conta_dias_uteis

df_sla = pd.DataFrame({
    'protocolo': ['P001',       'P002',       'P003',       'P004'],
    'abertura':  ['2026-06-02', '2026-06-05', None,         '2026-06-10'],
    'resolucao': ['2026-06-08', '2026-06-04', '2026-06-15', '2026-06-09'],
    # P001: seg->seg excluindo sab(06) + feriado(07) -> 4 dias uteis
    # P002: fim < inicio -> NaN
    # P003: abertura None -> NaN
    # P004: 1 dia util
})

feriados = ['2026-06-07']  # Corpus Christi

df_sla['dias_uteis'] = Conta_dias_uteis(
    pd.to_datetime(df_sla['abertura']),
    pd.to_datetime(df_sla['resolucao']),
    holidays=feriados,
    include_end=True,
)

print(df_sla)


## 7. `contagem_ativos_por_dia`

Gera a série temporal de quantos itens (contratos, projetos, chamados) estão ativos em cada dia usando soma cumulativa de eventos de abertura e fechamento.

In [ ]:
from ML.supervisionado.series_temporais.transformacoes_temporais import contagem_ativos_por_dia

df_contratos = pd.DataFrame({
    'id':    ['C1',          'C2',          'C3',          'C4'],
    'inicio':['2026-06-01', '2026-06-03', '2026-06-02', '2026-06-05'],
    'fim':   ['2026-06-05', '2026-06-07', '2026-06-06', '2026-06-09'],
})

df_serie = contagem_ativos_por_dia(df_contratos, col_inicio='inicio', col_fim='fim')

print(df_serie)

df_serie.plot(x='data', y='ativos', title='Contratos ativos por dia',
              figsize=(9, 3), marker='o', color='steelblue')
plt.tight_layout()
plt.show()
